<a href="https://colab.research.google.com/github/BigamPachhai/DeepLearning/blob/main/Next_word_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/qoute_dataset.csv")

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [ ]:
df.shape

(3038, 2)

In [ ]:
quotes=df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
quotes=quotes.str.lower()

In [ ]:
import string
translator=str.maketrans('', '', string.punctuation)
quotes=quotes.apply(lambda x: x.translate(translator))

In [ ]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [ ]:
vocab_size=10000
tokenizer=Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [ ]:
word_index=tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [ ]:
sequence=tokenizer.texts_to_sequences(quotes)

In [ ]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [ ]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [ ]:
X=[]
y=[]

for seq in sequence:
  for i in range(1, len(seq)):
    X.append(seq[:i])
    y.append(seq[i])

In [ ]:
len(X)

85271

In [ ]:
len(y)

85271

In [ ]:
max_len=max(len(x) for x in X)
print(max_len)

745


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded=pad_sequences(X,maxlen=max_len)

In [ ]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [ ]:
y=np.array(y)

In [ ]:
X_padded.shape

(85271, 745)

In [ ]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [ ]:
y.shape

(85271,)

In [ ]:
y_one_hot.shape

(85271, 10000)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN, LSTM, Dense

In [ ]:
embedding_dim=50
rnn_units=128

In [ ]:
rnn_model=Sequential()
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))

rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))


In [ ]:
lstm_model.compile(
    optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs=10
batch_size=128


In [ ]:
history_rnn=rnn_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size,validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 50s 74ms/step - accuracy: 0.0460 - loss: 6.7358 - val_accuracy: 0.0201 - val_loss: 10.7530
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0696 - loss: 6.2747 - val_accuracy: 0.0816 - val_loss: 6.4563
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0936 - loss: 5.9481 - val_accuracy: 0.0950 - val_loss: 6.3460
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1067 - loss: 5.7062 - val_accuracy: 0.1004 - val_loss: 6.3654
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1178 - loss: 5.5130 - val_accuracy: 0.1041 - val_loss: 6.3377
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1293 - loss: 5.3315 - val_accuracy: 0.1065 - val_loss: 6.3352
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1381 - loss: 5.1392 - val_accuracy: 0.1080 - val_loss: 6.3900
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 42s 71ms/step - accuracy: 0.1486 - loss: 4.9616 -

In [ ]:
epochs=100
batch_size=128
history_lstm=lstm_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size,validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 56ms/step - accuracy: 0.0390 - loss: 6.7576 - val_accuracy: 0.0469 - val_loss: 6.6880
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.0587 - loss: 6.3275 - val_accuracy: 0.0655 - val_loss: 6.5520
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0819 - loss: 6.0474 - val_accuracy: 0.0829 - val_loss: 6.4648
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0989 - loss: 5.8250 - val_accuracy: 0.0946 - val_loss: 6.4318
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 56ms/step - accuracy: 0.1108 - loss: 5.6349 - val_accuracy: 0.1001 - val_loss: 6.4113
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1214 - loss: 5.4583 - val_accuracy: 0.1061 - val_loss: 6.4151
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1296 - loss: 5.3006 - val_accuracy: 0.1074 - val_loss: 6.4447
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1380 - loss: 5

In [ ]:
lstm_model.save('lstm_model.h5')

In [ ]:
index_to_word={}
for word,index in word_index.items():
  index_to_word[index]=word

In [ ]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [ ]:
seed_text = "what are "
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

the


In [ ]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text


In [ ]:
seed = "Meaning of life"
generated_output = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generated_output)

Meaning of life is the only options left and laughing feels better to


In [ ]:
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)